# 🍏 Apple Generative Imagery Systems - Studio v2 (Native OpenEXR)
### 3D Spatial LookDev Pipeline (Direct Depth.exr + Normal.exr + Color ID.exr + Custom LoRA)
---
이 주피터 노트북은 Houdini의 **32-bit Depth.exr, Normal.exr, Id.exr 원본 파일을 변환 없이 직접 ComfyUI에서 로딩**하는 Native EXR 파이프라인입니다.

In [ ]:
# 1. GPU 사양 확인
!nvidia-smi

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. ComfyUI 최신 엔진 + Native OpenEXR 커스텀 노드 자동 설치
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || true
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q opencv-python-headless openexr

# 📦 [핵심] 32-bit EXR을 변환 없이 실시간 로딩하는 Native EXR 커스텀 노드 생성
os_custom_node_dir = "/content/ComfyUI/custom_nodes/comfyui_openexr_loader"
import os
os.makedirs(os_custom_node_dir, exist_ok=True)

with open(os.path.join(os_custom_node_dir, "load_exr.py"), "w") as f:
    f.write('''
import os, torch, numpy as np
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import cv2

class LoadNativeEXR:
    @classmethod
    def INPUT_TYPES(s):
        return {
            "required": {
                "folder_path": ("STRING", {"default": "/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/3d_guides/0000"}),
                "exr_file_name": ("STRING", {"default": "Depth.exr"}),
                "pass_type": (["Depth", "Normal", "Color_ID", "Direct_RGB"], {"default": "Depth"}),
            }
        }

    RETURN_TYPES = ("IMAGE",)
    RETURN_NAMES = ("IMAGE",)
    FUNCTION = "load_and_process_exr"
    CATEGORY = "3D_VFX_Pipeline"

    def load_and_process_exr(self, folder_path, exr_file_name, pass_type):
        full_path = os.path.join(folder_path, exr_file_name)
        if not os.path.exists(full_path):
            candidates = [f for f in os.listdir(folder_path) if pass_type.lower() in f.lower() and f.endswith(".exr")]
            if candidates:
                full_path = os.path.join(folder_path, candidates[0])
            else:
                raise FileNotFoundError(f"❌ EXR 파일을 찾을 수 없습니다: {full_path}")

        print(f"📦 [LoadNativeEXR] 32-bit EXR 실시간 직접 로딩: {full_path} ({pass_type})")
        img = cv2.imread(full_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise ValueError(f"❌ EXR 파일 읽기 실패: {full_path}")

        if pass_type == "Depth":
            d_val = img[:, :, 0] if len(img.shape) == 3 else img
            valid = (d_val > 0.001) & (d_val < 1000.0) & (~np.isnan(d_val)) & (~np.isinf(d_val))
            if np.any(valid):
                d_min, d_max = np.percentile(d_val[valid], 1), np.percentile(d_val[valid], 99)
                norm_d = np.clip((d_val - d_min) / (d_max - d_min + 1e-6), 0.0, 1.0)
                d_norm = (1.0 - norm_d)
                d_norm[~valid] = 0.0
            else:
                d_norm = np.zeros_like(d_val, dtype=np.float32)
            rgb = np.stack([d_norm, d_norm, d_norm], axis=-1)
        elif pass_type == "Normal":
            rgb = np.clip((img + 1.0) * 0.5, 0.0, 1.0)
        else:
            rgb = np.clip(img, 0.0, 1.0)

        if rgb.shape[0] != 1080 or rgb.shape[1] != 1920:
            interp = cv2.INTER_NEAREST if pass_type == "Color_ID" else cv2.INTER_LANCZOS4
            rgb = cv2.resize(rgb, (1920, 1080), interpolation=interp)

        tensor = torch.from_numpy(rgb.astype(np.float32)).unsqueeze(0)
        return (tensor,)

NODE_CLASS_MAPPINGS = {"LoadNativeEXR": LoadNativeEXR}
NODE_DISPLAY_NAME_MAPPINGS = {"LoadNativeEXR": "📦 Load 3D EXR Pass (Native)"}
''')

print('✅ ComfyUI 및 Native OpenEXR 커스텀 노드 준비 완료!')

In [ ]:
# 4. SDXL Base + VAE + ControlNet (Depth & Normal) 가중치 다운로드
print('🚀 필수 AI 모델 가중치 초고속 다운로드 중...')
# 1) SDXL Base 1.0 (6.4GB)
!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P /content/ComfyUI/models/checkpoints/

# 2) SDXL VAE Fix (335MB)
!wget -c https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors -P /content/ComfyUI/models/vae/

# 3) SDXL ControlNet Depth (2.5GB)
!wget -c https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O /content/ComfyUI/models/controlnet/controlnet-depth-sdxl-1.0.safetensors

# 4) SDXL ControlNet Normal / Union (2.3GB)
!wget -c https://huggingface.co/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors -O /content/ComfyUI/models/controlnet/controlnet-normal-sdxl-1.0.safetensors

# 5) LoRA 가중치 자동 심볼릭 링크
lora_target_dir = "/content/ComfyUI/models/loras"
os.makedirs(lora_target_dir, exist_ok=True)
lora_found = glob.glob("/content/drive/MyDrive/**/apple_minimal_craft_sdxl_v1.safetensors", recursive=True)
if lora_found:
    src_lora = lora_found[0]
    dst_lora = os.path.join(lora_target_dir, "apple_minimal_craft_sdxl_v1.safetensors")
    if not os.path.exists(dst_lora):
        os.symlink(src_lora, dst_lora)
    print(f'✅ LoRA 연동 완료: {src_lora}')

print('✅ 모든 모델 준비 완료!')

In [ ]:
# 5. 🌐 Localtunnel & Cloudflare를 통한 ComfyUI 웹 실행
!npm install -g localtunnel
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import subprocess, threading, time, urllib.request

def run_comfy():
    !python /content/ComfyUI/main.py --listen 127.0.0.1 --port 8188 --enable-cors-header --highvram --dont-upcast-attention

threading.Thread(target=run_comfy, daemon=True).start()
time.sleep(5)

try:
    public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
except:
    public_ip = '확인 불가'

print('='*75)
print('🚀 [방법 1] Localtunnel 접속 링크 (가장 안정적 ⭐)')
print(f'🔑 Localtunnel 비밀번호(Password): {public_ip}')
print('='*75)

!npx localtunnel --port 8188 &

time.sleep(3)
print('\n' + '='*75)
print('🚀 [방법 2] Cloudflare 접속 링크')
print('='*75)
!cloudflared tunnel --url http://127.0.0.1:8188